# Infera — fine-tune NLI on real SciFact (GPU, class-weighted)

This fixes the REFUTES regression from the earlier CPU fine-tuning run: SciFact's train split has roughly half as many contradiction (REFUTES) examples as entailment/neutral, so plain fine-tuning taught the model to under-predict contradiction. This notebook uses inverse-frequency class weights to counteract that.

**Before running anything: Runtime -> Change runtime type -> T4 GPU, then Save.**

Run the cells top to bottom. At the end it downloads the fine-tuned checkpoint as a zip — drop it into `backend/data/scifact-nli-weighted/` locally and point `INFERA_NLI_MODEL` at that folder.

In [ ]:
!nvidia-smi --query-gpu=name --format=csv,noheader
!pip install -q "transformers>=4.45" "accelerate>=1.1.0" scikit-learn

## Fetch the real SciFact release (same source as the local training script)

In [ ]:
!curl -s -o data.tar.gz https://scifact.s3-us-west-2.amazonaws.com/release/latest/data.tar.gz
!tar xzf data.tar.gz
!ls data

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path

DATA_DIR = Path("data")
_SCIFACT_TO_NLI = {"SUPPORT": "entailment", "CONTRADICT": "contradiction", "NOINFO": "neutral"}
LABEL_LIST = ["entailment", "neutral", "contradiction"]

@dataclass
class Example:
    premise: str
    hypothesis: str
    label: str

def _load_corpus():
    corpus = {}
    with open(DATA_DIR / "corpus.jsonl") as f:
        for line in f:
            doc = json.loads(line)
            corpus[doc["doc_id"]] = " ".join(doc["abstract"])
    return corpus

def load_examples(split: str) -> list[Example]:
    corpus = _load_corpus()
    examples = []
    with open(DATA_DIR / f"claims_{split}.jsonl") as f:
        for line in f:
            claim = json.loads(line)
            evidence = claim.get("evidence") or {}
            for doc_id in claim.get("cited_doc_ids") or []:
                abstract = corpus.get(doc_id)
                if not abstract:
                    continue
                doc_evidence = evidence.get(str(doc_id))
                label = _SCIFACT_TO_NLI[doc_evidence[0]["label"] if doc_evidence else "NOINFO"]
                examples.append(Example(premise=abstract, hypothesis=claim["claim"], label=label))
    return examples

train_examples = load_examples("train")
val_examples = load_examples("dev")
print(f"train={len(train_examples)} val={len(val_examples)}")

## Baseline: score the pretrained (not yet fine-tuned) checkpoint on dev — this is the "before" number

In [ ]:
BASE_MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

from sklearn.metrics import classification_report, confusion_matrix

def score(gold, preds):
    report = classification_report(gold, preds, labels=LABEL_LIST, output_dict=True, zero_division=0)
    matrix = confusion_matrix(gold, preds, labels=LABEL_LIST)
    return {"accuracy": report["accuracy"], "macro_f1": report["macro avg"]["f1-score"],
            "per_class": {l: report[l] for l in LABEL_LIST}, "confusion_matrix": matrix.tolist()}

def print_report(name, result):
    print(f"\n=== {name} ===")
    print(f"accuracy={result['accuracy']:.3f}  macro_f1={result['macro_f1']:.3f}")
    print(f"{'label':<14}{'precision':>10}{'recall':>10}{'f1':>10}{'support':>10}")
    for label, m in result["per_class"].items():
        print(f"{label:<14}{m['precision']:>10.3f}{m['recall']:>10.3f}{m['f1-score']:>10.3f}{int(m['support']):>10}")
    print("confusion matrix (rows=gold, cols=predicted):", LABEL_LIST)
    for label, row in zip(LABEL_LIST, result["confusion_matrix"]):
        print(f"  {label:<14}{row}")

from transformers import pipeline

baseline_clf = pipeline("text-classification", model=BASE_MODEL, top_k=None, device=0)
inputs = [{"text": e.premise, "text_pair": e.hypothesis} for e in val_examples]
raw = baseline_clf(inputs, batch_size=16, truncation="only_first", max_length=256)
preds = [max(scores, key=lambda o: o["score"])["label"].lower() for scores in raw]
gold = [e.label for e in val_examples]
baseline_result = score(gold, preds)
print_report("BASELINE (pretrained, not fine-tuned)", baseline_result)

del baseline_clf
import torch, gc
gc.collect(); torch.cuda.empty_cache()

## Fine-tune with class-weighted loss

In [ ]:
import numpy as np
import torch
from torch import nn
from datasets import Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments

OUTPUT_DIR = "./scifact-nli-weighted"
# Back to 4 epochs, matching the original (successful, non-collapsed) CPU
# run exactly -- the 8-epoch attempt wasn't the fix. What actually broke the
# last run: epoch 1 trained fine (loss 1.074), then from epoch 2 onward
# training loss was exactly 0.0 and eval accuracy froze at an identical
# value for 7 straight epochs -- gradients underflowed to zero in fp16 and
# the model silently stopped updating. The model likely auto-loaded in fp16
# on GPU without Trainer's fp16 flag being set, so PyTorch's usual loss-
# scaling safety net for fp16 training never activated. Forcing float32
# below removes that failure mode entirely, matching how the CPU run (which
# never collapsed) trained.
EPOCHS = 4
BATCH_SIZE = 16
LR = 1e-5
MAX_LENGTH = 256

label_counts = np.array([sum(e.label == l for e in train_examples) for l in LABEL_LIST])
class_weights = torch.tensor(len(train_examples) / (len(LABEL_LIST) * label_counts), dtype=torch.float32)
print(f"class weights ({LABEL_LIST}): {class_weights.tolist()}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABEL_LIST), ignore_mismatched_sizes=True, torch_dtype=torch.float32,
)
print(f"model dtype: {next(model.parameters()).dtype}")  # must print torch.float32, not float16/bfloat16

# Force a fresh classifier head: most 3-way NLI checkpoints already have 3
# labels so the shape matches and `ignore_mismatched_sizes` keeps the
# pretrained head as-is -- but that head's own label order often isn't
# entailment/neutral/contradiction, so silently reusing it scrambles a
# working head into a broken one. Reset it and train from scratch on SciFact
# with our label order; the encoder underneath keeps its pretrained knowledge.
reset_count = 0
for name, module in model.named_modules():
    if name.endswith("classifier") and isinstance(module, torch.nn.Linear):
        module.reset_parameters()
        reset_count += 1
        print(f"reset classifier head: {name} ({module})")
assert reset_count > 0, "no classifier Linear module matched by name -- the reset never ran, which reproduces the scrambled-head bug"
model.config.id2label = dict(enumerate(LABEL_LIST))
model.config.label2id = {l: i for i, l in enumerate(LABEL_LIST)}

def to_hf_dataset(examples):
    return Dataset.from_list([{"premise": e.premise, "hypothesis": e.hypothesis, "label": e.label} for e in examples])

def tokenize(batch):
    enc = tokenizer(batch["premise"], batch["hypothesis"], truncation="only_first", max_length=MAX_LENGTH)
    enc["labels"] = [LABEL_LIST.index(l) for l in batch["label"]]
    return enc

remove_cols = ["premise", "hypothesis", "label"]
train_ds = to_hf_dataset(train_examples).map(tokenize, batched=True, remove_columns=remove_cols)
val_ds = to_hf_dataset(val_examples).map(tokenize, batched=True, remove_columns=remove_cols)

steps_per_epoch = -(-len(train_examples) // BATCH_SIZE)
total_steps = steps_per_epoch * EPOCHS
warmup_steps = max(10, int(0.1 * total_steps))
print(f"steps_per_epoch={steps_per_epoch} total_steps={total_steps} warmup_steps={warmup_steps}")

def compute_metrics(eval_pred):
    from sklearn.metrics import f1_score
    preds = np.argmax(eval_pred.predictions, axis=1)
    acc = (preds == eval_pred.label_ids).mean()
    macro_f1 = f1_score(eval_pred.label_ids, preds, average="macro")
    return {"accuracy": float(acc), "macro_f1": float(macro_f1)}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=20,
    report_to=[],
    fp16=False,
    bf16=False,
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        weight = class_weights.to(device=outputs.logits.device, dtype=outputs.logits.dtype)
        loss = nn.functional.cross_entropy(outputs.logits, labels, weight=weight)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    model=model, args=training_args, train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, compute_metrics=compute_metrics,
)
trainer.train()

## Score the fine-tuned model on dev — compare against the baseline above, especially the contradiction row

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

finetuned_clf = pipeline("text-classification", model=OUTPUT_DIR, top_k=None, device=0)
raw = finetuned_clf(inputs, batch_size=16, truncation="only_first", max_length=MAX_LENGTH)
preds = [max(scores, key=lambda o: o["score"])["label"].lower() for scores in raw]
finetuned_result = score(gold, preds)
print_report("FINE-TUNED (class-weighted)", finetuned_result)

print("\n=== Summary: baseline vs fine-tuned ===")
print(f"accuracy:  {baseline_result['accuracy']:.3f} -> {finetuned_result['accuracy']:.3f}")
print(f"macro F1:  {baseline_result['macro_f1']:.3f} -> {finetuned_result['macro_f1']:.3f}")
b, f = baseline_result["per_class"]["contradiction"], finetuned_result["per_class"]["contradiction"]
print(f"REFUTES (contradiction) F1:  {b['f1-score']:.3f} -> {f['f1-score']:.3f}")
print(f"REFUTES (contradiction) recall:  {b['recall']:.3f} -> {f['recall']:.3f}")

## Download the checkpoint — unzip locally into `backend/data/scifact-nli-weighted/` and set `INFERA_NLI_MODEL` to that path

In [ ]:
import shutil
shutil.make_archive("scifact-nli-weighted", "zip", OUTPUT_DIR)

from google.colab import files
files.download("scifact-nli-weighted.zip")